# 05: Evaluate LLaVA + LoRA (H3 Ablation)

Loads the LoRA adapter trained in notebook 04 onto the LLaVA backbone and runs
ORA on the synthetic held-out set + a Mind2Web slice. Compares against the
non-adapted ORA numbers from notebook 03.

**Settings:** Accelerator = `GPU T4 x2`, Internet = `On`.

**Datasets to attach:**
- `gvla-weights`
- `gvla-data`
- `gvla-lora-r1` (output of notebook 04)
- `gvla-runs-YYYY-MM-DD` (output of notebook 03; for comparison)

In [ ]:
import os, subprocess
WEIGHTS = '/kaggle/input/gvla-weights/hf_cache'
DATA    = '/kaggle/input/gvla-data'
ADAPTER = '/kaggle/input/gvla-lora-r1'
os.environ['HF_HOME'] = WEIGHTS
os.environ['TRANSFORMERS_OFFLINE'] = '1'
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'])

In [ ]:
REPO_URL = 'https://github.com/<your-org>/grounded_vla.git'
!if [ ! -d /kaggle/working/grounded_vla ]; then git clone {REPO_URL} /kaggle/working/grounded_vla; fi
!git -C /kaggle/working/grounded_vla pull
%cd /kaggle/working/grounded_vla
!pip install -q -e .[gpu]

In [ ]:
# Load LLaVA + adapter manually so we can inject PEFT before the agent runs.
from grounded_vla.backends.llava import LLaVABackend

class LLaVAWithLoRA(LLaVABackend):
    name = 'llava-1.6-7b+lora'
    def __init__(self, adapter_dir, **kw):
        super().__init__(**kw)
        self.adapter_dir = adapter_dir
    def _ensure_loaded(self):
        if self._model is not None:
            return
        super()._ensure_loaded()
        from peft import PeftModel
        self._model = PeftModel.from_pretrained(self._model, self.adapter_dir)
        self._model.eval()
        print(f'[LLaVAWithLoRA] adapter loaded from {self.adapter_dir}')

In [ ]:
from grounded_vla.agents import ORAAgent
from grounded_vla.backends.base import GenerationConfig
from grounded_vla.data import make_dataset
from grounded_vla.eval import EvalRunner
from pathlib import Path

backend = LLaVAWithLoRA(
    adapter_dir=ADAPTER,
    model_id=f'{WEIGHTS}/llava-v1.6-mistral-7b-hf',
    device='cuda',
    quantize='4bit',
)
agent = ORAAgent(backend, GenerationConfig(max_new_tokens=256, temperature=0.1))

RUNS = Path('/kaggle/working/runs')

def run(ds_name, dataset):
    out = RUNS / f'ora_lora__{ds_name}'
    res = EvalRunner(agent).evaluate(
        dataset, save_dir=out, checkpoint_every=5, resume=True
    )
    print(f'ora_lora on {ds_name}: '
          f'success={res.task_completion_rate:.3f} '
          f'mean_steps={res.mean_steps:.2f} '
          f'errors={res.error_breakdown}')
    return res

synthetic = make_dataset({
    'kind': 'jsonl',
    'path':  f'{DATA}/synthetic_sample/synthetic_sample.jsonl',
    'source': 'synthetic',
})
import glob
_m2w_jsonls = sorted(glob.glob(f'{DATA}/mind2web/*.jsonl'))
if not _m2w_jsonls:
    raise FileNotFoundError(f'no Mind2Web JSONL under {DATA}/mind2web/')
mind2web = make_dataset({
    'kind': 'mind2web',
    'jsonl_path': _m2w_jsonls[0],
    'images_dir':  f'{DATA}/mind2web/images',
    'limit': 30,
})

_ = run('synthetic', synthetic)
_ = run('mind2web',  mind2web)
backend.close()

In [ ]:
# Compare ORA-base vs ORA+LoRA side by side (assumes notebook 03 results are
# attached as `gvla-runs-YYYY-MM-DD`; otherwise swap in your own paths).
import glob, json
base_runs = sorted(glob.glob('/kaggle/input/gvla-runs-*/runs/ora_llava__*'))
lora_runs = sorted(glob.glob('/kaggle/working/runs/ora_lora__*'))

def load(path):
    return json.loads(open(f'{path}/summary.json').read())

print(f'{"dataset":12s} {"ora-base":>10s} {"ora+lora":>10s} {"delta":>8s}')
for lp in lora_runs:
    ds = lp.rsplit('__', 1)[-1]
    base = next((p for p in base_runs if p.endswith(ds)), None)
    if not base:
        continue
    b = load(base)['task_completion_rate']
    l = load(lp)['task_completion_rate']
    print(f'{ds:12s} {b:10.3f} {l:10.3f} {l-b:+8.3f}')

## Interpreting the result

- **Positive delta on synthetic, smaller on Mind2Web:** LoRA closed the in-domain
  gap (expected); cross-domain transfer is partial — useful talking point in §6.
- **Negative delta:** likely overfitting on a tiny synthetic set. Either grow the
  synthetic corpus to ~200 reviewed examples, or shorten training (1 epoch).
- **No change:** LoRA rank may be too small (try `r=32`) or the action format in
  the synthetic dataset doesn't match the agent's prompt template.

Either way, the result goes into the H3 row of the results table in the report.